GRU(Gated Recurrent Unit)

In [1]:
import pandas as pd

df=pd.read_csv('../datasets/news_dataset.csv')
df.head()

,label,text
0,REAL,Payal has accused filmmaker Anurag Kashyap of ...
1,FAKE,A four-minute-long video of a woman criticisin...
2,FAKE,"Republic Poll, a fake Twitter account imitatin..."
3,REAL,"Delhi teen finds place on UN green list, turns..."
4,REAL,Delhi: A high-level meeting underway at reside...


In [2]:
df.isna().sum()

label    0
text     8
dtype: int64

In [3]:
#Drop rows where 'text' is NaN
df.dropna(subset=['text'],inplace=True)

In [4]:
df.isna().sum()

label    0
text     0
dtype: int64

In [5]:
df["label"].value_counts()

label
FAKE    1871
REAL    1850
Name: count, dtype: int64

In [6]:
df["text"] = df["text"].str.lower()

In [7]:
import re
def clean_text(text):
    text=re.sub(r"https?://\S+|www\.\S+"," ",text)
    text=re.sub(r"@\w+"," ",text)
    text=re.sub(r"#"," ",text)
    text=re.sub(r"&\w+"," ",text)
    text=re.sub(r"\d+"," ",text)
    text=re.sub(r"\s+"," ",text).strip()

    return text

In [8]:
df["text"]=df["text"].apply(clean_text)

In [17]:
from nltk.stem import PorterStemmer

stemmer=PorterStemmer()
df['tokens'] = df['tokens'].apply(lambda toks: [stemmer.stem(t) for t in toks])
df

,label,text,tokens
0,REAL,payal has accused filmmaker anurag kashyap of ...,"[p, a, y, a, l, , h, a, s, , a, c, c, u, s, ..."
1,FAKE,a four-minute-long video of a woman criticisin...,"[a, , f, o, u, r, -, m, i, n, u, t, e, -, l, ..."
2,FAKE,"republic poll, a fake twitter account imitatin...","[r, e, p, u, b, l, i, c, , p, o, l, l, , ,, ..."
3,REAL,"delhi teen finds place on un green list, turns...","[d, e, l, h, i, , t, e, e, n, , f, i, n, d, ..."
4,REAL,delhi: a high-level meeting underway at reside...,"[d, e, l, h, i, , :, , a, , h, i, g, h, -, ..."
...,...,...,...
3724,REAL,: (ist) sep the second round of countrywide se...,"[:, , (, , i, s, t, , ), , s, e, p, , t, ..."
3725,REAL,: (ist) sep the second round of countrywide se...,"[:, , (, , i, s, t, , ), , s, e, p, , t, ..."
3726,FAKE,the bengaluru city police’s official twitter h...,"[t, h, e, , b, e, n, g, a, l, u, r, u, , c, ..."
3727,REAL,"sep , , : am ist source: toi.in meet neelkanth...","[s, e, p, , ,, , ,, , :, , a, m, , i, s, ..."


In [18]:
from nltk.stem import WordNetLemmatizer

lemmatizer= WordNetLemmatizer()

# Lemmatize each token
df["tokens"]=df["tokens"].apply(lambda toks: [lemmatizer.lemmatize(t) for t in toks])

# Join the tokens back into a sentence
df['tokens'] = df["tokens"].apply(' '.join)
df

,label,text,tokens
0,REAL,payal has accused filmmaker anurag kashyap of ...,p a y a l h a s a c c u s e d f i l m m ...
1,FAKE,a four-minute-long video of a woman criticisin...,a f o u r - m i n u t e - l o n g v i d e ...
2,FAKE,"republic poll, a fake twitter account imitatin...","r e p u b l i c p o l l , a f a k e ..."
3,REAL,"delhi teen finds place on un green list, turns...",d e l h i t e e n f i n d s p l a c e ...
4,REAL,delhi: a high-level meeting underway at reside...,d e l h i : a h i g h - l e v e l m e ...
...,...,...,...
3724,REAL,: (ist) sep the second round of countrywide se...,: ( i s t ) s e p t h e s e c o n ...
3725,REAL,: (ist) sep the second round of countrywide se...,: ( i s t ) s e p t h e s e c o n ...
3726,FAKE,the bengaluru city police’s official twitter h...,t h e b e n g a l u r u c i t y p o l i ...
3727,REAL,"sep , , : am ist source: toi.in meet neelkanth...","s e p , , : a m i s t s o u r c e ..."


In [19]:
from sklearn.preprocessing import LabelEncoder

label_eoncoder=LabelEncoder()
y_encoded=label_eoncoder.fit_transform(df["label"])
print(label_eoncoder.classes_)

['FAKE' 'REAL']


In [20]:
y_encoded

array([1, 0, 0, ..., 0, 1, 1], shape=(3721,))

In [21]:
num_class=len(label_eoncoder.classes_)
print(num_class)

2


In [22]:
df["tokens"]

0       p a y a l   h a s   a c c u s e d   f i l m m ...
1       a   f o u r - m i n u t e - l o n g   v i d e ...
2       r e p u b l i c   p o l l   ,   a   f a k e   ...
3       d e l h i   t e e n   f i n d s   p l a c e   ...
4       d e l h i   :   a   h i g h - l e v e l   m e ...
                              ...                        
3724    :   (   i s t   )   s e p   t h e   s e c o n ...
3725    :   (   i s t   )   s e p   t h e   s e c o n ...
3726    t h e   b e n g a l u r u   c i t y   p o l i ...
3727    s e p   ,   ,   :   a m   i s t   s o u r c e ...
3728    r e a d   a l s o   r e a d   a l s o   a d v ...
Name: tokens, Length: 3721, dtype: str

In [23]:
all_words = " ".join(df["tokens"]).split()

In [24]:
unique_words = sorted(list(set(all_words)))
print(unique_words)

['!', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', ':', ';', '<', '=', '>', '?', '@', '[', ']', '_', '`', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '{', '|', '}', '~', '£', '¦', '©', '«', '®', '°', '·', '¹', 'º', '×', 'à', 'á', 'â', 'ã', 'ä', 'å', 'ç', 'è', 'é', 'ì', 'í', 'î', 'ï', 'ñ', 'ò', 'ó', 'ö', 'ú', 'û', 'ü', 'ý', 'þ', 'ā', 'ď', 'ğ', 'ō', 'ş', 'ʳ', 'ʸ', 'ʿ', 'ˢ', '̶', 'α', 'υ', 'а', 'б', 'в', 'г', 'д', 'е', 'и', 'й', 'к', 'л', 'м', 'н', 'о', 'п', 'р', 'с', 'т', 'ц', 'ч', 'ы', 'э', 'я', 'є', 'ѕ', '،', 'ء', 'آ', 'أ', 'ؤ', 'إ', 'ئ', 'ا', 'ب', 'ة', 'ت', 'ث', 'ج', 'ح', 'خ', 'د', 'ذ', 'ر', 'ز', 'س', 'ش', 'ص', 'ض', 'ط', 'ظ', 'ع', 'غ', 'ـ', 'ف', 'ق', 'ك', 'ل', 'م', 'ن', 'ه', 'و', 'ى', 'ي', 'َ', 'ُ', 'ِ', 'ّ', 'ْ', 'ٙ', 'ٹ', 'پ', 'چ', 'ڈ', 'ڌ', 'ڑ', 'ک', 'گ', 'ں', 'ھ', 'ہ', 'ی', 'ے', '۔', 'ँ', 'ं', 'ः', 'अ', 'आ', 'इ', 'ई', 'उ', 'ऊ', 'ऋ', 'ए', 'ऐ', 'ऑ', 'ऒ', 'ओ', 'औ', 'क', 'ख', 'ग', 'घ',

In [25]:
vocab={word:i+1 for i, word in enumerate(unique_words)}
vocab_size=len(vocab)+1
print(vocab_size)

880


In [26]:
print(vocab)

{'!': 1, '$': 2, '%': 3, '&': 4, "'": 5, '(': 6, ')': 7, '*': 8, '+': 9, ',': 10, '-': 11, '.': 12, '/': 13, ':': 14, ';': 15, '<': 16, '=': 17, '>': 18, '?': 19, '@': 20, '[': 21, ']': 22, '_': 23, '`': 24, 'a': 25, 'b': 26, 'c': 27, 'd': 28, 'e': 29, 'f': 30, 'g': 31, 'h': 32, 'i': 33, 'j': 34, 'k': 35, 'l': 36, 'm': 37, 'n': 38, 'o': 39, 'p': 40, 'q': 41, 'r': 42, 's': 43, 't': 44, 'u': 45, 'v': 46, 'w': 47, 'x': 48, 'y': 49, 'z': 50, '{': 51, '|': 52, '}': 53, '~': 54, '£': 55, '¦': 56, '©': 57, '«': 58, '®': 59, '°': 60, '·': 61, '¹': 62, 'º': 63, '×': 64, 'à': 65, 'á': 66, 'â': 67, 'ã': 68, 'ä': 69, 'å': 70, 'ç': 71, 'è': 72, 'é': 73, 'ì': 74, 'í': 75, 'î': 76, 'ï': 77, 'ñ': 78, 'ò': 79, 'ó': 80, 'ö': 81, 'ú': 82, 'û': 83, 'ü': 84, 'ý': 85, 'þ': 86, 'ā': 87, 'ď': 88, 'ğ': 89, 'ō': 90, 'ş': 91, 'ʳ': 92, 'ʸ': 93, 'ʿ': 94, 'ˢ': 95, '̶': 96, 'α': 97, 'υ': 98, 'а': 99, 'б': 100, 'в': 101, 'г': 102, 'д': 103, 'е': 104, 'и': 105, 'й': 106, 'к': 107, 'л': 108, 'м': 109, 'н': 110, 'о': 11

In [27]:
X_sequences=[]
for sentence in df["tokens"]: #to remove duplicate value
    seq=[vocab[word] for word in sentence.split() if word in vocab]
    X_sequences.append(seq)

In [28]:
for i , seq in enumerate(X_sequences):
    print(seq) #vocab id

[40, 25, 49, 25, 36, 32, 25, 43, 25, 27, 27, 45, 43, 29, 28, 30, 33, 36, 37, 37, 25, 35, 29, 42, 25, 38, 45, 42, 25, 31, 35, 25, 43, 32, 49, 25, 40, 39, 30, 26, 29, 32, 25, 46, 33, 38, 31, 33, 38, 25, 40, 40, 42, 39, 40, 42, 33, 25, 44, 29, 36, 49, 47, 33, 44, 32, 32, 29, 42, 33, 38, 25, 46, 33, 28, 29, 39, 44, 32, 25, 44, 47, 29, 38, 44, 46, 33, 42, 25, 36, 12, 43, 32, 29, 37, 25, 33, 38, 44, 25, 33, 38, 29, 28, 32, 29, 42, 43, 44, 25, 38, 27, 29, 47, 32, 33, 36, 29, 43, 40, 29, 25, 35, 33, 38, 31, 44, 39, 29, 44, 33, 37, 29, 43, 25, 38, 28, 43, 25, 33, 28, 10, 598, 33, 32, 25, 46, 29, 47, 25, 38, 44, 29, 28, 44, 39, 43, 40, 29, 25, 35, 25, 26, 39, 45, 44, 44, 32, 33, 43, 30, 39, 42, 25, 36, 39, 38, 31, 44, 33, 37, 29, 12, 26, 45, 44, 44, 39, 28, 25, 49, 10, 30, 33, 38, 25, 36, 36, 49, 33, 44, 32, 39, 45, 31, 32, 44, 33, 37, 45, 43, 44, 31, 29, 44, 33, 44, 39, 30, 30, 37, 49, 32, 29, 25, 28, 12, 33, 32, 25, 28, 44, 47, 29, 29, 44, 29, 28, 25, 26, 39, 45, 44, 37, 49, 33, 38, 27, 33, 28

In [29]:
import numpy as np

max_len=100
x_padded=np.zeros((len(X_sequences),max_len),dtype=np.int32)
print(x_padded)

[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


In [30]:
for i, seq in enumerate(X_sequences):
    if len(seq) > 0:
        x_padded[i, : len(seq[:max_len])] = seq[:max_len]

In [31]:
print(x_padded)

[[40 25 49 ... 38 44 25]
 [25 30 39 ... 33 29 25]
 [42 29 40 ... 39 38 44]
 ...
 [44 32 29 ... 45 44 44]
 [43 29 40 ... 39 38 33]
 [42 29 25 ... 42 28 33]]


In [32]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test=train_test_split(x_padded,y_encoded,test_size=0.2,random_state=42,stratify=y_encoded)
print(x_train)

[[ 25  40  42 ...  33  36  33]
 [ 38  29  47 ...  39  42  42]
 [ 25  38  33 ...  44  39  27]
 ...
 [203 242 197 ...  36  31  25]
 [ 43  29  46 ...  44  25  42]
 [ 25  30  25 ...  29  43  25]]


In [33]:
import torch
from torch.utils.data import DataLoader,TensorDataset

train_dataset=TensorDataset(torch.tensor(x_train,dtype=torch.long),torch.tensor(y_train,dtype=torch.long))

test_dataset=TensorDataset(torch.tensor(x_test,dtype=torch.long),torch.tensor(y_test,dtype=torch.long))

In [34]:
batch_size=64
train_loader=DataLoader(train_dataset,batch_size=batch_size,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=batch_size,shuffle=False)

In [35]:
device=torch.device("cuda" if torch.cuda.is_available() else"cpu")
print(f"using device: {device}")

using device: cpu


In [36]:
import torch.nn as nn

class SimpleGRU(nn.Module):

    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        self.gru = nn.GRU(
            embedding_dim,
            hidden_dim,
            batch_first=True
        )

        self.fc = nn.Linear(
            hidden_dim,
            output_dim
        )

    def forward(self, text):

        embedded = self.embedding(text)

        out, hidden = self.gru(embedded)

        final_hidden = hidden[-1]

        return self.fc(final_hidden)

In [37]:
EMBEDDING_DIM=100
HIDDEN_DIM=128

vocab_size = len(vocab) + 1

model = SimpleGRU(
    vocab_size,
    EMBEDDING_DIM,
    HIDDEN_DIM,
    num_class
).to(device)


In [38]:
criterion=nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [39]:
epochs=5

for epoch in range(epochs):
    model.train()
    
    epoch_loss = 0
    correct = 0
    total = 0
    
    for texts,labels in train_loader:
        texts = texts.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        predictions=model(texts)
        
        loss=criterion(predictions,labels)
        
        loss.backward()
        
        optimizer.step()
        
        epoch_loss +=loss.item()
        _,predicted=torch.max(predictions.data,1)
        total+=labels.size(0)
        correct+=(predicted==labels).sum().item()
        
    accuracy=100*correct/total
    print(
        f"Epoch {epoch+1:02d}/{epochs:02d} | "
        f"Loss: {epoch_loss/len(train_loader):.4f} | "
        f"Train Accuracy: {accuracy:.2f}%"
    )

Epoch 01/05 | Loss: 0.6212 | Train Accuracy: 65.62%
Epoch 02/05 | Loss: 0.4547 | Train Accuracy: 80.61%
Epoch 03/05 | Loss: 0.3219 | Train Accuracy: 87.43%
Epoch 04/05 | Loss: 0.2430 | Train Accuracy: 89.95%
Epoch 05/05 | Loss: 0.1970 | Train Accuracy: 92.10%


In [40]:
model.eval()
test_correct=0
test_total=0

with torch.no_grad():
    for texts,labels in test_loader:
        texts=texts.to(device)
        labels = labels.to(device)

        predictions = model(texts)
        _, predicted = torch.max(predictions.data, 1)
        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()

test_accuracy = 100 * test_correct / test_total
print(f"\nFinal Test Accuracy: {test_accuracy:.2f}%")


Final Test Accuracy: 91.14%
